# Safe BO実践：Mixed変数とq-batchの安全な候補提案

このNotebookでは、連続・整数・カテゴリ変数を含む探索空間で、**複数候補を同時に提案し、各候補の予測上の安全性と変数の合法性を確認する**方法を学びます。

最初に説明用の軽量な予測モデルでMixed×q-batchの契約を確認し、後半でrobotorchanのGPを `fit()` して安全候補を提案する例を示します。既存の `SafeBOWorkflow` と `SafetySpecification` を再利用します。

**先に読む：** [Notebook 50：Safe BOの基本](50_safe_bo_usage_guide.ipynb)。安全と予測された候補も、実機での安全を保証するものではありません。


## 1. Mixed変数・q-batch・安全制約の違い

| 要素 | このNotebookの設定 | 判定内容 |
|---|---|---|
| 連続変数 | `x0`：0〜1 | 実数の入力 |
| 整数変数 | `x1`：0,1,2,3 | 小数値を提案しない |
| カテゴリ変数 | `x2`：0,2,4 | 許されたカテゴリ値のみ |
| 安全出力 | `g(X)=x0-0.8` | `g(X)<=0` を満たすか |
| q-batch | `q=3` | 3行の候補をまとめて提案 |

**q=3は入力次元が3という意味ではありません。** ここでは入力次元 `d=3` なので結果の形が偶然 `[3,3]` になります。`q` と `d` は独立した概念です。

In [ ]:
import torch

from robotorchan.optim.domains.variable_space import MixedVariableSpace
from robotorchan.optim.safe_workflow import SafeBOWorkflow
from robotorchan.semantics import ContinuousConstraint, SafetySpecification

torch.set_default_dtype(torch.double)
bounds = torch.tensor([[0.0, 0.0, 0.0], [1.0, 3.0, 4.0]])
space = MixedVariableSpace(
    bounds=bounds,
    integer_dims=(1,),
    categorical_values={2: (0.0, 2.0, 4.0)},
)


def safety_output(X):
    return X[..., :1] - 0.8


class DemoPosterior:
    def __init__(self, X):
        self.mean = safety_output(X)
        self.variance = torch.full_like(self.mean, 0.0001)


class DemoSafetyModel:
    def posterior(self, X):
        return DemoPosterior(X)


class DemoBatchAcquisition:
    def __call__(self, X):
        return X[..., 0].sum(dim=-1)


safety_model = DemoSafetyModel()
acquisition = DemoBatchAcquisition()

### 簡易モデルを使う理由

`DemoSafetyModel` は `posterior(X)` の `mean` と `variance` を返す説明用モデルです。安全判定とMixed候補生成を独立に確認するために使います。

`DemoBatchAcquisition` も**実際のBO獲得関数ではなく**、バッチ内の連続変数を合計するだけのデモ用関数です。ここで性能やGPの学習効果を評価することはできません。

後半では既存の `SingleTaskGP.fit()` を使う例を追加します。

## 1. 安全な初期データと3点のバッチ提案

この例では安全出力を`g(X) = X[..., 0] - 0.8`とし、`g(X) <= 0`を安全条件とします。安全性は1列目の連続変数に依存し、他の列は整数・カテゴリの合法値を守る必要があります。

- `initial_X`：安全と確認した最初の観測点
- `SafetySpecification`：安全出力の閾値、初期観測、信頼係数を指定
- `SafeBOWorkflow`：候補の提案（`ask`）と観測値の登録（`tell`）を管理
- `q=3`：3点を同時に提案
- `num_samples=256`：評価する候補バッチ数

`DemoBatchAcquisition`はバッチ内の連続変数値の合計を評価する簡単な関数です。実運用では、q対応の獲得関数を利用します。

候補の形は`[3, 3]`、安全出力の観測値の形は`[3, 1]`です。バッチ内の**全点が予測上安全**であることを確認してから採用します。

In [ ]:
initial_X = torch.tensor([[0.1, 1.0, 2.0]])
specification = SafetySpecification(
    constraints=(ContinuousConstraint(0, 0.0),),
    initial_X=initial_X,
    observed_constraints=safety_output(initial_X),
    beta=4.0,
)
workflow = SafeBOWorkflow(specification, bounds, variable_space=space)

batch = workflow.ask(
    safety_model,
    mode="objective",
    acq_function=acquisition,
    q=3,
    num_samples=256,
    seed=7,
)
print("提案バッチ:", batch)
assert batch.shape == (3, 3)
assert bool((batch[:, 1] == batch[:, 1].round()).all())
assert bool(torch.isin(batch[:, 2], torch.tensor([0.0, 2.0, 4.0])).all())
assert bool((safety_output(batch) + 0.02 <= 0.0).all())

observed_constraints = safety_output(batch)
workflow.tell(batch, observed_constraints)
print("観測登録後の件数:", workflow.observed_X.shape[0])

### 提案バッチを可視化する

横軸は連続変数、縦軸は整数変数です。点の横にカテゴリ値を表示し、**安全制約と整数・カテゴリの合法性を同時に満たしている**ことを確認します。

この合成例では真の安全境界が `x0=0.8` と分かっていますが、実際のSafe BOでは安全境界は未知です。

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(batch[:, 0], batch[:, 1], s=100, label="Proposed safe batch")
for row in batch:
    ax.annotate(f"cat={int(row[2].item())}", (row[0].item(), row[1].item()), xytext=(5, 6), textcoords="offset points")
ax.axvline(0.8, linestyle="--", color="gray", label="True safety boundary (demo only)")
ax.set(xlim=(0, 1), ylim=(-0.3, 3.3), yticks=[0, 1, 2, 3], xlabel="Continuous x0", ylabel="Integer x1", title="Mixed q-batch candidates")
ax.legend()
plt.show()

### `tell()` が登録するのは実測した安全出力

`workflow.tell(batch, observed_constraints)` は、バッチの各行に対応する安全出力を登録します。**予測平均や獲得関数値を観測値として登録してはいけません**。

この例では `safety_output()` が測定装置の代わりです。`tell()` だけでGPが自動再学習されるわけではありません。

## 2. 安全領域内の探索（expansion）

`mode="objective"`では獲得関数が高いバッチを選びます。一方、`mode="expansion"`では**安全と予測される領域の中で、安全モデルの不確実性が高い候補**を選びます。

ここでは`q=2`に変更するので、結果は2行×3列です。先ほどの観測と同じく、実測した安全出力を`tell()`で登録します。

注意：この簡易モデルの予測分散は一定なので、探索モードは不確実性の学習効果を示す実験ではありません。SafeOptの厳密な到達可能集合の計算でもありません。

In [ ]:
expansion_batch = workflow.ask(
    safety_model,
    mode="expansion",
    q=2,
    num_samples=256,
    seed=8,
)
print("探索バッチ:", expansion_batch)
assert expansion_batch.shape == (2, 3)
workflow.tell(expansion_batch, safety_output(expansion_batch))
print("観測登録後の件数:", workflow.observed_X.shape[0])

### `objective` と `expansion` の比較

`mode="objective"` は、予測上安全なバッチの中から獲得関数が高い候補を選びます。`mode="expansion"` は、安全と予測される範囲で安全モデルの不確実性を重視します。

ただしこの簡易モデルは分散が一定なので、`expansion` の**学習効果を示す例ではありません**。実測安全出力を登録した後にGPを学習する場合は、後半の例を参照してください。

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(batch[:, 0], batch[:, 1], s=100, label="Objective mode (q=3)")
ax.scatter(expansion_batch[:, 0], expansion_batch[:, 1], marker="x", s=100, label="Expansion mode (q=2)")
ax.axvline(0.8, linestyle="--", color="gray", label="True safety boundary (demo only)")
ax.set(xlim=(0, 1), ylim=(-0.3, 3.3), yticks=[0, 1, 2, 3], xlabel="Continuous x0", ylabel="Integer x1", title="Two Safe BO modes")
ax.legend()
plt.show()

In [ ]:
from botorch.acquisition.analytic import PosteriorMean
from robotorchan.models import SingleTaskGP

continuous_bounds = torch.tensor([[0.0], [1.0]])
initial_continuous_X = torch.tensor([[0.05], [0.12], [0.20], [0.32]])

def continuous_safety(X):
    return X - 0.8

def continuous_objective(X):
    return -(X - 0.45).square()

continuous_spec = SafetySpecification(
    constraints=(ContinuousConstraint(0, 0.0),),
    initial_X=initial_continuous_X,
    observed_constraints=continuous_safety(initial_continuous_X),
    beta=4.0,
)
continuous_workflow = SafeBOWorkflow(continuous_spec, continuous_bounds)
continuous_objective_model = SingleTaskGP(initial_continuous_X, continuous_objective(initial_continuous_X))
continuous_objective_model.fit()
continuous_objective_model.eval()
continuous_safety_model = SingleTaskGP(initial_continuous_X, continuous_safety(initial_continuous_X))
continuous_safety_model.fit()
continuous_safety_model.eval()
continuous_candidate = continuous_workflow.ask(
    continuous_safety_model, mode="objective",
    acq_function=PosteriorMean(continuous_objective_model),
    num_samples=256, seed=15,
)
print("GP-based safe proposal:", continuous_candidate.item())

### 実測値の登録・再学習・次の候補提案

`ask()` で提案した候補を実験し、安全出力を `tell()` に登録します。目的関数の観測値も別途保存し、**最新の観測データで両方のGPを再学習**します。

この例は連続変数のGP学習手順を示すもので、Mixed×q-batchの全組み合わせをGPで検証したという意味ではありません。

In [ ]:
continuous_observed_g = continuous_safety(continuous_candidate)
continuous_observed_y = continuous_objective(continuous_candidate)
continuous_workflow.tell(continuous_candidate, continuous_observed_g)
continuous_updated_X = torch.cat([initial_continuous_X, continuous_candidate], dim=0)
continuous_updated_Y = torch.cat([continuous_objective(initial_continuous_X), continuous_observed_y], dim=0)
continuous_updated_G = torch.cat([continuous_safety(initial_continuous_X), continuous_observed_g], dim=0)
updated_objective_model = SingleTaskGP(continuous_updated_X, continuous_updated_Y)
updated_objective_model.fit()
updated_objective_model.eval()
updated_safety_model = SingleTaskGP(continuous_updated_X, continuous_updated_G)
updated_safety_model.fit()
updated_safety_model.eval()
next_continuous_candidate = continuous_workflow.ask(
    updated_safety_model, mode="objective",
    acq_function=PosteriorMean(updated_objective_model),
    num_samples=256, seed=16,
)
print("Observed candidate:", continuous_candidate.item())
print("Next proposal:", next_continuous_candidate.item())

### GPの安全予測と提案位置を可視化する

安全GPの予測平均と±2標準偏差、真の安全閾値、実測済みの入力、次の未評価候補を表示します。

**±2標準偏差の帯は説明用の可視化**であり、`SafeBOWorkflow` 内部の安全判定を独自に再実装したものではありません。

In [ ]:
grid = torch.linspace(0, 1, 201).unsqueeze(-1)
with torch.no_grad():
    posterior = updated_safety_model.posterior(grid)
    gp_mean = posterior.mean.squeeze(-1).detach()
    gp_sd = posterior.variance.clamp_min(0).sqrt().squeeze(-1).detach()
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(grid.squeeze(-1), gp_mean, label="Safety GP mean")
ax.fill_between(grid.squeeze(-1), gp_mean-2*gp_sd, gp_mean+2*gp_sd, alpha=0.2, label="Mean ± 2 SD")
ax.axhline(0.0, linestyle="--", color="gray", label="Safety threshold")
ax.scatter(continuous_updated_X.squeeze(-1), continuous_updated_G.squeeze(-1), label="Observed safety outputs")
ax.axvline(next_continuous_candidate.item(), linestyle=":", label="Next proposal")
ax.set(xlabel="Continuous x", ylabel="Safety output", title="GP-based Safe BO")
ax.legend()
plt.show()

## 3. 学習済みGPを使うSafe BO

ここからは、**連続変数だけの小さな例**で `SingleTaskGP` を実際に学習します。Mixed×q-batchのデモ用モデルとは切り分け、まずGPの事後分布を用いた安全判定の動作を確かめます。

Mixed変数で実際のGPを利用する場合は、変数型に対応したモデルとq対応獲得関数を組み合わせる必要があります。カテゴリを連続値として扱うだけでは、カテゴリ間の距離を正しくモデル化できるとは限りません。

## 3. 学習済みGPを使う場合

このNotebookの`DemoSafetyModel`は、`posterior(X)`から予測平均と分散を返す**説明用のモデル**です。実際のBOでは、robotorchanの`SingleTaskGP`や、カテゴリを含む場合は`MixedSingleTaskGP`など、入力構造に適したGPを学習して置き換えます。

- GP学習：安全出力の観測データでモデルを作成し、`fit()`で学習
- 候補提案：学習済み安全GPとq対応獲得関数を`ask()`に渡す
- 実験：提案されたすべての条件を測定
- 観測登録：安全出力を`tell()`へ渡し、目的関数の観測値も別に保存
- モデル更新：新しい観測を追加してGPと獲得関数を作り直す

**実装上の範囲：** Mixed×q>1は有限個の候補バッチをJoint評価する方式です。候補が見つからない場合は例外となり、危険な候補への自動フォールバックは行いません。実測安全性の保証や非同期バッチ制御は含みません。

## 4. まとめ：どのAPIが何を担当するか

| 目的 | 既存API・要素 | 注意点 |
|---|---|---|
| Mixed変数の合法値 | `MixedVariableSpace` | 整数とカテゴリを区別 |
| 安全出力の閾値 | `ContinuousConstraint` | `g(X)<=0` の符号に注意 |
| 初期安全観測 | `SafetySpecification` | 事前に安全確認した条件を使用 |
| q-batchの安全候補 | `SafeBOWorkflow.ask(q=...)` | **全候補が予測上安全**である必要 |
| 実測値の登録 | `SafeBOWorkflow.tell()` | 予測値ではなく実測安全出力 |
| GP学習 | `SingleTaskGP.fit()` | 最新観測データで再学習 |
| 目的関数の改善 | `PosteriorMean` など | q対応の獲得関数は別途選択 |

**注意：** Mixed×q-batchは有限候補バッチのJoint評価です。候補が見つからなければ例外となり、危険な候補へ自動フォールバックしません。実測安全性の保証、非同期バッチ制御、厳密なSafeOpt到達可能集合の計算は含みません。

基本の安全制約付きBOは [Notebook 50](50_safe_bo_usage_guide.ipynb) を参照してください。